In [17]:
from pathlib import Path
import pandas as pd

In [18]:
RAW = Path("data")
FILES = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "translation": "product_category_name_translation.csv",
}
dfs = {k: pd.read_csv(RAW / v) for k, v in FILES.items()}


In [19]:
# 1. Общий профиль каждой таблицы: shape, дубликаты, пропуски, dtypes
for name, df in dfs.items():
    print(f"\n=== {name}: shape={df.shape}, full-row duplicates={df.duplicated().sum()}")
    prof = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "nulls": df.isna().sum(),
        "null_%": (df.isna().mean() * 100).round(2),
        "nunique": df.nunique(),
    })
    print(prof)

orders, items, pay = dfs["orders"], dfs["items"], dfs["payments"]
cust, prod, trans = dfs["customers"], dfs["products"], dfs["translation"]



=== customers: shape=(99441, 5), full-row duplicates=0
                           dtype  nulls  null_%  nunique
customer_id               object      0     0.0    99441
customer_unique_id        object      0     0.0    96096
customer_zip_code_prefix   int64      0     0.0    14994
customer_city             object      0     0.0     4119
customer_state            object      0     0.0       27

=== orders: shape=(99441, 8), full-row duplicates=0
                                dtype  nulls  null_%  nunique
order_id                       object      0    0.00    99441
customer_id                    object      0    0.00    99441
order_status                   object      0    0.00        8
order_purchase_timestamp       object      0    0.00    98875
order_approved_at              object    160    0.16    90733
order_delivered_carrier_date   object   1783    1.79    81018
order_delivered_customer_date  object   2965    2.98    95664
order_estimated_delivery_date  object      0    0.00 

In [20]:
# 2. Диапазон дат и распределение статусов (отмены)
date_cols = [c for c in orders.columns if c.endswith(("_timestamp", "_date", "_at"))]
orders[date_cols] = orders[date_cols].apply(pd.to_datetime, errors="coerce")
print("\nДиапазон order_purchase_timestamp:",
      orders.order_purchase_timestamp.min(), "→", orders.order_purchase_timestamp.max())
print("\nЗаказов по месяцам (проверка разреженности краёв):")
print(orders.groupby(orders.order_purchase_timestamp.dt.to_period("M")).size())
print("\nСтатусы заказов:")
print(orders.order_status.value_counts(dropna=False).to_frame("n")
      .assign(share=lambda d: (d.n / d.n.sum()).round(4)))



Диапазон order_purchase_timestamp: 2016-09-04 21:15:19 → 2018-10-17 17:30:18

Заказов по месяцам (проверка разреженности краёв):
order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, dtype: int64

Статусы заказов:
                  n   share
order_status               
delivered     96478  0.9702
shipped        1107  0.0111
canceled        625  0.0063
unavailable     609  0.0061
invoiced        314  0.0032
processing      301  0.0030
created           5  0.0001
approved          2  0.0000


In [21]:
# 3. Отрицательные / нулевые значения
print("\nprice <= 0:", (items.price <= 0).sum(),
      "| freight_value < 0:", (items.freight_value < 0).sum(),
      "| payment_value <= 0:", (pay.payment_value <= 0).sum())
print(items[["price", "freight_value"]].describe())



price <= 0: 0 | freight_value < 0: 0 | payment_value <= 0: 9
               price  freight_value
count  112650.000000  112650.000000
mean      120.653739      19.990320
std       183.633928      15.806405
min         0.850000       0.000000
25%        39.900000      13.080000
50%        74.990000      16.260000
75%       134.900000      21.150000
max      6735.000000     409.680000


In [22]:
# 4. Клиенты: customer_id vs customer_unique_id
print("\ncustomer_id unique:", cust.customer_id.nunique(),
      "| customer_unique_id unique:", cust.customer_unique_id.nunique())
orders_per_client = orders.merge(cust, on="customer_id").groupby("customer_unique_id").size()
print("Доля клиентов с >=2 заказами (все статусы):",
      round((orders_per_client >= 2).mean(), 4))
print("Заказов без customer_id в customers:",
      (~orders.customer_id.isin(cust.customer_id)).sum())



customer_id unique: 99441 | customer_unique_id unique: 96096
Доля клиентов с >=2 заказами (все статусы): 0.0312
Заказов без customer_id в customers: 0


In [23]:
# 5. Категории: пропуски и покрытие переводом
cat_items = items.merge(prod[["product_id", "product_category_name"]], on="product_id", how="left")
print("\nПозиций без категории:", cat_items.product_category_name.isna().sum(),
      f"({cat_items.product_category_name.isna().mean():.2%})")
no_tr = set(prod.product_category_name.dropna()) - set(trans.product_category_name)
print("Категории без английского перевода:", no_tr)

# 6. Целостность связей
print("\nitems.order_id отсутствует в orders:", (~items.order_id.isin(orders.order_id)).sum())
print("orders без items:", (~orders.order_id.isin(items.order_id)).sum())
print("Дубликаты (order_id, order_item_id):",
      items.duplicated(["order_id", "order_item_id"]).sum())

# 7. Логика дат: delivered без даты доставки, доставка раньше покупки
dl = orders[orders.order_status == "delivered"]
print("\ndelivered без order_delivered_customer_date:",
      dl.order_delivered_customer_date.isna().sum())
print("доставка раньше покупки:",
      (orders.order_delivered_customer_date < orders.order_purchase_timestamp).sum())

# 8. Сверка двух определений выручки на delivered-заказах
rev_items = items[items.order_id.isin(dl.order_id)][["price", "freight_value"]].sum()
rev_pay = pay[pay.order_id.isin(dl.order_id)].payment_value.sum()
print("\nSUM(price):", round(rev_items.price, 2),
      "| price+freight:", round(rev_items.sum(), 2),
      "| SUM(payment_value):", round(rev_pay, 2))


Позиций без категории: 1603 (1.42%)
Категории без английского перевода: {'portateis_cozinha_e_preparadores_de_alimentos', 'pc_gamer'}

items.order_id отсутствует в orders: 0
orders без items: 775
Дубликаты (order_id, order_item_id): 0

delivered без order_delivered_customer_date: 8
доставка раньше покупки: 0

SUM(price): 13221498.11 | price+freight: 15419773.75 | SUM(payment_value): 15422461.77
